In [1]:
# generating embeddings
import os
import cv2
import numpy as np

from tqdm import tqdm
from insightface.app import FaceAnalysis

INPUT_ROOT = "dataset_split/test/probe_cropped"
OUTPUT_ROOT = "embeddings/test/probe_embeddings"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("Loading ArcFace...")

app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0, det_size=(640, 640))

rec_model = app.models["recognition"]

print("ArcFace Ready")

count = 0

for identity in tqdm(os.listdir(INPUT_ROOT)):

    input_dir = os.path.join(INPUT_ROOT, identity)
    output_dir = os.path.join(OUTPUT_ROOT, identity)

    os.makedirs(output_dir, exist_ok=True)

    for image_file in os.listdir(input_dir):

        image_path = os.path.join(input_dir, image_file)

        img = cv2.imread(image_path)

        if img is None:
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img = cv2.resize(img, (112, 112))

        embedding = rec_model.get_feat(img).flatten().astype(np.float32)

        save_path = os.path.join(output_dir, os.path.splitext(image_file)[0] + ".npy")

        np.save(save_path, embedding)

        count += 1

print("Total Embeddings:", count)

Loading ArcFace...
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}


/opt/miniconda3/lib/python3.13/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:149: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'CoreMLExecutionProvider, AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


find model: /Users/admin/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)
ArcFace Ready


100%|██████████| 30/30 [00:12<00:00,  2.33it/s]

Total Embeddings: 135


In [2]:
# creating dataset to feed to model
import os
import torch
import pyiqa
import numpy as np
import pandas as pd

from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

ROOT_IMAGES = "dataset_split"
ROOT_EMBEDDINGS = "embeddings"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

musiq_metric = pyiqa.create_metric("musiq", device=device)


def get_quality_score(image_path):

    score = musiq_metric(image_path)

    return float(score.cpu().item())


def load_gallery_db(gallery_root):

    gallery_db = {}

    for identity in os.listdir(gallery_root):

        gallery_db[identity] = []

        identity_dir = os.path.join(gallery_root, identity)

        for file in os.listdir(identity_dir):

            emb = np.load(os.path.join(identity_dir, file))

            gallery_db[identity].append(emb)

    return gallery_db


def search_gallery(probe_embedding, gallery_db):

    scores = {}

    for identity in gallery_db:

        sims = []

        for gallery_emb in gallery_db[identity]:

            sim = cosine_similarity(
                probe_embedding.reshape(1, -1), gallery_emb.reshape(1, -1)
            )[0][0]

            sims.append(sim)

        scores[identity] = max(sims)

    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    predicted_identity = sorted_scores[0][0]

    best_similarity = sorted_scores[0][1]

    second_similarity = sorted_scores[1][1]

    margin = best_similarity - second_similarity

    return (predicted_identity, best_similarity, second_similarity, margin)


gallery_db = load_gallery_db("embeddings/test/gallery")

rows = []

probe_emb_root = "embeddings/test/probe_embeddings"
probe_img_root = "dataset_split/test/probe_cropped"

for identity in tqdm(os.listdir(probe_emb_root)):

    emb_dir = os.path.join(probe_emb_root, identity)

    img_dir = os.path.join(probe_img_root, identity)

    for emb_file in os.listdir(emb_dir):

        probe_embedding = np.load(os.path.join(emb_dir, emb_file))

        base_name = os.path.splitext(emb_file)[0]

        image_path = None

        for ext in [".jpg", ".jpeg", ".png"]:

            candidate = os.path.join(img_dir, base_name + ext)

            if os.path.exists(candidate):

                image_path = candidate
                break

        if image_path is None:
            continue

        quality_score = get_quality_score(image_path)

        predicted_identity, best_similarity, second_similarity, margin = search_gallery(
            probe_embedding, gallery_db
        )

        label = int(predicted_identity == identity)

        rows.append(
            {
                "true_identity": identity,
                "predicted_identity": predicted_identity,
                "quality_score": quality_score,
                "best_similarity": best_similarity,
                "second_similarity": second_similarity,
                "margin": margin,
                "label": label,
            }
        )

clean_df = pd.DataFrame(rows)

clean_df.to_csv("clean_testdata.csv", index=False)

print(clean_df.shape)

Loading pretrained model MUSIQ from /Users/admin/.cache/torch/hub/pyiqa/musiq_koniq_ckpt-e95806b9.pth


100%|██████████| 30/30 [00:25<00:00,  1.19it/s]

(135, 7)


In [7]:
# running svm
import joblib

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

svm = joblib.load("svm_rejection_model.pkl")

scaler = joblib.load("svm_scaler.pkl")

df = pd.read_csv("clean_testdata.csv")

X = df[["quality_score", "best_similarity", "margin"]]

X_scaled = scaler.transform(X)

df["svm_decision"] = svm.predict(X_scaled)

df["confidence_score"] = svm.predict_proba(X_scaled)[:, 1]

y_true = df["label"]
y_pred = df["svm_decision"]

print(classification_report(y_true, y_pred))

cm = confusion_matrix(y_true, y_pred)

print("\nConfusion Matrix")
print(cm)

TN = cm[0][0]
FP = cm[0][1]
FN = cm[1][0]
TP = cm[1][1]

TRR = TN / (TN + FP)
FAR = FP / (TN + FP)

TAR = TP / (TP + FN)
FRR = FN / (TP + FN)

print("\nSVM RESULTS")

print("Accuracy:", accuracy_score(y_true, y_pred))

print("TRR:", round(TRR, 4))
print("FAR:", round(FAR, 4))
print("TAR:", round(TAR, 4))
print("FRR:", round(FRR, 4))

              precision    recall  f1-score   support

           0       0.40      0.67      0.50         3
           1       0.99      0.98      0.98       132

    accuracy                           0.97       135
   macro avg       0.70      0.82      0.74       135
weighted avg       0.98      0.97      0.97       135


Confusion Matrix
[[  2   1]
 [  3 129]]

SVM RESULTS
Accuracy: 0.9703703703703703
TRR: 0.6667
FAR: 0.3333
TAR: 0.9773
FRR: 0.0227


In [4]:
# fixed threshold method
THRESHOLD = 0.40

threshold_pred = (df["best_similarity"] >= THRESHOLD).astype(int)

print(classification_report(y_true, threshold_pred))

cm_threshold = confusion_matrix(y_true, threshold_pred)

print(cm_threshold)

TN = cm_threshold[0][0]
FP = cm_threshold[0][1]
FN = cm_threshold[1][0]
TP = cm_threshold[1][1]

threshold_TRR = TN / (TN + FP)
threshold_FAR = FP / (TN + FP)

threshold_TAR = TP / (TP + FN)
threshold_FRR = FN / (TP + FN)

              precision    recall  f1-score   support

           0       0.60      1.00      0.75         3
           1       1.00      0.98      0.99       132

    accuracy                           0.99       135
   macro avg       0.80      0.99      0.87       135
weighted avg       0.99      0.99      0.99       135

[[  3   0]
 [  2 130]]


In [5]:
# final comparison table
comparison = pd.DataFrame(
    {
        "Method": ["Fixed Threshold", "SVM"],
        "Accuracy": [
            accuracy_score(y_true, threshold_pred),
            accuracy_score(y_true, y_pred),
        ],
        "TRR": [threshold_TRR, TRR],
        "FAR": [threshold_FAR, FAR],
        "TAR": [threshold_TAR, TAR],
        "FRR": [threshold_FRR, FRR],
    }
)

print(comparison)

            Method  Accuracy       TRR       FAR       TAR       FRR
0  Fixed Threshold  0.985185  1.000000  0.000000  0.984848  0.015152
1              SVM  0.970370  0.666667  0.333333  0.977273  0.022727
